# Solar Operations Business Intelligence

## Business Problem

A residential solar company wants to understand which territories are the most profitable. They want insight into their highest-quality customers so that they can make decisions that can impact their business and drive revenue. Sales and operational data are fragmented across different systems. Management is not in sync with sales representatives, and data is fragmented across different systems.

# Goal
Build a prototype that:

1. Generates (synthetic) solar customer data 
2. Creates a unified view of our customer journey
3. Analyzes prospect quality by territory and lead source
4. Identifies actionable insights to support business decisions

# Key Question
**Which territories and lead sources generate the highest-quality customers, and where should the company focus its resources?**

Our generated leads:

In [0]:
%python

import pandas as pd
import numpy as np

from datetime import datetime, timedelta

# Reproducibility: same results every time we run this cell
np.random.seed(42)

# Define our business dimensions
NUM_LEADS = 1000

territories = ["The Bronx", "White Plains", "Scarsdale", "Rockland County", "Yonkers", "Greenville"]

lead_sources = [
    "Door-to-door",
    "Referral",
    "Vendor-lead"
]

homeowner_types = [
    "Single-family",
    "Multi-family"
]

sales_reps = [
    "Rep A", "Rep B", "Rep C",
    "Rep D", "Rep E"
]

print("Business dimensions initialized.")
print(f"Generating {NUM_LEADS} prospective customers.")



In [0]:
%python
# Assign probabilities to each territory and lead source
territory_probs = [0.1027, 0.1712, 0.1370, 0.2123, 0.1712, 0.2056]
source_probs = [0.45, 0.10, 0.45]

start_date = datetime(2026, 1, 1)

leads = pd.DataFrame({
    "lead_id": [f"L{i:04d}" for i in range(1, NUM_LEADS + 1)],

    "territory": np.random.choice(
        territories, NUM_LEADS, p=territory_probs
    ),

    "lead_source": np.random.choice(
        lead_sources, NUM_LEADS, p=source_probs
    ),

    "homeowner_type": np.random.choice(
        homeowner_types, NUM_LEADS, p=[0.75, 0.25]
    ),

    "monthly_electric_bill": np.random.normal(
        loc=250, scale=65, size=NUM_LEADS
    ).clip(70, 600).round(2),

    "appt_date": [
        start_date + timedelta(days=int(d))
        for d in np.random.randint(0, 270, NUM_LEADS)
    ]
})

print(f"Created {len(leads)} leads.")
display(leads.head(10))


In [0]:

# Base qualification & contract percentages by lead source
qualification_rates = {
    "Door-to-door": 0.65,
    "Referral": 0.40,
    "Vendor-lead": 0.30
}

contract_rates = {
    "Door-to-door": 0.35,
    "Referral": 0.50,
    "Vendor-lead": 0.20
}

# Modifies performance by territory
territory_qualification_adjustment = {
    "The Bronx": -0.08, 
    "White Plains": 0.00, 
    "Scarsdale": -0.02, 
    "Rockland County": 0.05, 
    "Yonkers": 0.00, 
    "Greenville": 0.08
}

# Modify contract rate by territory
territory_contract_adjustment = {
    "The Bronx": 0.06, 
    "White Plains": 0.03, 
    "Scarsdale": 0.0, 
    "Rockland County": 0.05, 
    "Yonkers": 0.00, 
    "Greenville": 0.04
}

sales = leads[["lead_id", "lead_source", "territory"]].copy()

sales["sales_rep"] = np.random.choice(
    sales_reps, size=len(sales)
)

qualification_probabilities = [
    np.clip(
        qualification_rates[source]
        + territory_qualification_adjustment[territory],
        0, 1
    )
    for source, territory in zip(
        sales["lead_source"], sales["territory"]
    )
]

sales["qualified"] = [
    np.random.random() < qualification_rates[source]
    for source in sales["lead_source"]
]

sales["appointment_outcome"] = np.where(
    sales["qualified"], "Sat", "Cancelled"
)

sales["contract_signed"] = [
    qualified and np.random.random() < contract_rates[source]
    for qualified, source in zip(
        sales["qualified"], sales["lead_source"]
    )
]

# Contract conversion is based on qualification
contract_probabilities = [
    np.clip(
        contract_rates[source]
        + territory_contract_adjustment[territory],
        0, 1
    )
    for source, territory in zip(
        sales["lead_source"], sales["territory"]
    )
]

sales["contract_value"] = np.where(
    sales["contract_signed"],
    np.random.normal(45000, 7000, len(sales)).clip(
        25000, 70000
    ).round(2),
    0
)

sales["cancellation_reason"] = [
    np.random.choice([
        "Financing",
        "Customer Withdrawal",
        "Roof Constraints",
        "Permitting",
        "Rescheduled"
    ])
    if status == "Cancelled"
    else None
    for status in sales["appointment_outcome"]
]

sales = sales.drop(columns=["lead_source", "territory"])

print(f"Created {len(sales)} sales records.")
display(sales.head(10))


In [0]:

# Only contracted customers enter installation
signed_customers = sales[
    sales["contract_signed"]
]["lead_id"].tolist()

installations = pd.DataFrame({
    "lead_id": signed_customers
})

installation_prob = 0.92

installations["installation_status"] = [
    "Completed" if np.random.random() < installation_prob
    else "Cancelled"
    for _ in signed_customers
]

installations["days_to_install"] = [
    int(np.random.normal(55, 17))
    if status == "Completed"
    else None
    for status in installations["installation_status"]
]

installations["cancellation_reason"] = [
    np.random.choice([
        "Customer Withdrawal",
        "Permitting"
    ])
    if status == "Cancelled"
    else None
    for status in installations["installation_status"]
]

print(f"Created {len(installations)} installation records.")
display(installations.head(10))


Validate data

In [0]:

print("DATASET VALIDATION")
print("-" * 40)

print("Leads:", len(leads))
print("Sales:", len(sales))
print("Installations:", len(installations))

print("\nDuplicate lead IDs:")
print(leads["lead_id"].duplicated().sum())

print("\nSales records without a matching lead:")
print(
    len(set(sales["lead_id"]) - set(leads["lead_id"]))
)

print("\nInstallation records without a signed contract:")
signed_ids = set(
    sales.loc[sales["contract_signed"], "lead_id"]
)
installation_ids = set(installations["lead_id"])

print(len(installation_ids - signed_ids))

print("\nContract count:")
print(sales["contract_signed"].sum())

print("\nInstallation completion rate:")
print(
    (installations["installation_status"] == "Completed")
    .mean()
)


# Converting datasets to Spark

In [0]:
# Check our active catalog and schema

spark.sql(
    """
    SELECT
      current_catalog() AS CATALOG,
      current_schema() AS SCHEMA
    """
).show()

In [0]:
# Convert pandas DF to spark DF
leads_df = spark.createDataFrame(leads)
sales_df = spark.createDataFrame(sales)
installs_df = spark.createDataFrame(installations)

print("Successfully converted pandats datasets to Spark")

print("Leads:")
leads_df.printSchema()
print("Sales:")
sales_df.printSchema()
print("Installations:")
installs_df.printSchema()

In [0]:
leads_df.write.format("delta").mode("overwrite").saveAsTable("solar_leads")
sales_df.write.format("delta").mode("overwrite").saveAsTable("solar_sales")
installs_df.write.format("delta").mode("overwrite").saveAsTable("solar_installations")

print("Successfully created our Delta tables")

In [0]:
%sql
SELECT 'leads' AS dataset, COUNT(*) AS records FROM solar_leads

UNION ALL 

SELECT 'sales', COUNT(*)
FROM solar_sales

UNION ALL

SELECT 'installations', COUNT(*)
FROM solar_installations;
